# 🎙️ کلون صدا و خواندن متن فارسی (OmniVoice) — نسخه‌ی بازنویسی‌شده

**روش کار:** یک فایل متنی (`txt`, `docx`, `md`, `srt`, `vtt`) + یک نمونه صدا (و متن همان نمونه) ← خواندن کل متن با صدای نمونه.

**قبل از شروع:**
1. از منوی `Runtime ← Change runtime type` یک **GPU** (مثلاً T4 رایگان) انتخاب کنید.
2. (اختیاری ولی توصیه‌شده) برای مدل فارسی، در Colab از منوی 🔑 **Secrets** یک مقدار با نام `HF_TOKEN` بسازید و توکن Hugging Face خود را (از نوع Read) در آن بگذارید. توکن را داخل خود نوت‌بوک ننویسید.
3. سلول‌ها را به ترتیب اجرا کنید.

**نکات مهم:**
- متن نمونه صدا را حتماً (به فارسی و دقیق) در سلول ۴ بنویسید.
- نمونه‌ی صدا باید فارسی، ۳ تا ۱۰ ثانیه، تمیز و بدون موسیقی باشد.
- تولید با «کش» انجام می‌شود: اگر Colab قطع شد یا چیزی را اصلاح کردید، فقط جمله‌های تغییرکرده دوباره ساخته می‌شوند.
- فقط از صدای خودتان یا کسانی استفاده کنید که اجازه‌ی کلون صدایشان را داده‌اند.

In [ ]:
#@title ⚙️ ۱) نصب و لود مدل
#@markdown `auto` یعنی اول مدل فارسی (C1Tech/OmniVoice_Persian_TTS) را امتحان کن و اگر لود نشد، مدل اصلی k2-fsa/OmniVoice را لود کن.
Model_Choice = "auto" #@param ["auto", "C1Tech/OmniVoice_Persian_TTS", "k2-fsa/OmniVoice"]

import os

!pip install -q omnivoice python-docx pydub soundfile

import torch

HAS_GPU = torch.cuda.is_available()
device = 'cuda:0' if HAS_GPU else 'cpu'
dtype = torch.float16 if HAS_GPU else torch.float32

if HAS_GPU:
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')
else:
    print('⚠️ GPU فعال نیست! روی CPU تولید صدا بسیار کند است.')
    print('   Runtime ← Change runtime type ← GPU را انتخاب و دوباره اجرا کنید.')

# توکن Hugging Face از Colab Secrets (نه از داخل کد)
def _get_hf_token():
    try:
        from google.colab import userdata
        t = userdata.get('HF_TOKEN')
        if t:
            return t.strip()
    except Exception:
        pass
    return os.environ.get('HF_TOKEN')

_tok = _get_hf_token()
if _tok:
    os.environ['HF_TOKEN'] = _tok
    from huggingface_hub import login
    login(token=_tok, add_to_git_credential=False)
    print('✅ توکن Hugging Face از Secrets خوانده شد.')
else:
    print('ℹ️ توکن HF_TOKEN پیدا نشد (برای مدل فارسی که نیاز به تأیید دسترسی دارد لازم است).')

from omnivoice import OmniVoice

candidates = (
    ['C1Tech/OmniVoice_Persian_TTS', 'k2-fsa/OmniVoice']
    if Model_Choice == 'auto' else [Model_Choice]
)

omni_model = None
LOADED_MODEL = None
for name in candidates:
    try:
        print(f'⏳ در حال لود {name} ...')
        omni_model = OmniVoice.from_pretrained(name, device_map=device, dtype=dtype)
        LOADED_MODEL = name
        break
    except Exception as e:
        print(f'⚠️ لود {name} ناموفق بود: {type(e).__name__}: {str(e)[:250]}')
        if 'Persian' in name:
            print('   (این مدل نیاز به تأیید دسترسی در صفحه‌ی Hugging Face و HF_TOKEN دارد.)')

if omni_model is None:
    raise RuntimeError('❌ هیچ مدلی لود نشد. اتصال اینترنت/توکن را بررسی کنید.')

print(f'✅ مدل لود شد: {LOADED_MODEL}')

In [ ]:
#@title 🧰 ۲) ابزارهای متن (نرمال‌سازی فارسی، تقسیم جمله، خواندن فایل)
import re, os, hashlib

# ---------- نرمال‌سازی حروف ----------
_CHAR_MAP = str.maketrans({
    'ي': 'ی', 'ى': 'ی', 'ك': 'ک', 'ة': 'ه', 'ـ': '',
    '\u200e': '', '\u200f': '', '\u202a': '', '\u202b': '', '\u202c': '',
    '\u202d': '', '\u202e': '', '\ufeff': '', '\xa0': ' ',
})
_DIGIT_MAP = str.maketrans('۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩', '01234567890123456789')

TAG_RE = re.compile(
    r'(\[(?:laughter|sigh|confirmation-en|question-(?:en|ah|oh|ei|yi)'
    r'|surprise-(?:ah|oh|wa|yo)|dissatisfaction-hnn)\])'
)

# ---------- اعداد به حروف فارسی ----------
_DIG = ['صفر', 'یک', 'دو', 'سه', 'چهار', 'پنج', 'شش', 'هفت', 'هشت', 'نه']
_UNITS = ['', 'یک', 'دو', 'سه', 'چهار', 'پنج', 'شش', 'هفت', 'هشت', 'نه', 'ده',
          'یازده', 'دوازده', 'سیزده', 'چهارده', 'پانزده', 'شانزده', 'هفده', 'هجده', 'نوزده']
_TENS = ['', '', 'بیست', 'سی', 'چهل', 'پنجاه', 'شصت', 'هفتاد', 'هشتاد', 'نود']
_HUNDS = ['', 'صد', 'دویست', 'سیصد', 'چهارصد', 'پانصد', 'ششصد', 'هفتصد', 'هشتصد', 'نهصد']
_SCALES = ['', 'هزار', 'میلیون', 'میلیارد', 'تریلیون']

def _three(n):
    parts = []
    h, r = divmod(n, 100)
    if h:
        parts.append(_HUNDS[h])
    if r:
        if r < 20:
            parts.append(_UNITS[r])
        else:
            t, u = divmod(r, 10)
            parts.append(_TENS[t])
            if u:
                parts.append(_UNITS[u])
    return ' و '.join(parts)

def num_to_fa(n):
    if n == 0:
        return 'صفر'
    if n < 0:
        return 'منفی ' + num_to_fa(-n)
    chunks = []
    i = 0
    while n:
        n, c = divmod(n, 1000)
        if c:
            s = _three(c)
            if i == 1 and c == 1:
                s = 'هزار'
            elif i > 0:
                s = s + ' ' + _SCALES[i]
            chunks.append(s)
        i += 1
    return ' و '.join(reversed(chunks))

def _digits_each(s):
    return ' '.join(_DIG[int(d)] for d in s if d.isdigit())

def _int_words(s):
    if len(s) > 15 or (len(s) > 1 and s[0] == '0'):
        return _digits_each(s)
    return num_to_fa(int(s))

def _dec_repl(m):
    whole = _int_words(m.group(1))
    frac = m.group(2)
    f = _int_words(frac) if (len(frac) <= 3 and not frac.startswith('0')) else _digits_each(frac)
    return f'{whole} ممیز {f}'

def _normalize_part(t):
    t = re.sub(r'https?://\S+|www\.\S+|\S+@\S+\.\S+', ' ', t)
    t = t.translate(_CHAR_MAP).translate(_DIGIT_MAP)
    t = re.sub(r'(?<=\d)[,٬](?=\d{3}(?!\d))', '', t)
    t = re.sub(r'[%٪]\s*(\d+(?:[.٫]\d+)?)', r'\1 درصد', t)
    t = re.sub(r'(\d+(?:[.٫]\d+)?)\s*[%٪]', r'\1 درصد', t)
    t = re.sub(r'(\d+)[.٫](\d+)', _dec_repl, t)
    t = re.sub(r'\d+', lambda m: _int_words(m.group(0)), t)
    t = t.replace('&', ' و ')
    t = re.sub(r'[«»“”"*_#`~|<>]', ' ', t)
    t = re.sub(r'\.{3,}', '…', t)
    t = re.sub(r'[ \t]+', ' ', t)
    return t

def normalize_text(text):
    parts = TAG_RE.split(text)
    out = [p if TAG_RE.fullmatch(p) else _normalize_part(p) for p in parts]
    return re.sub(r'\s+', ' ', ''.join(out)).strip()

# ---------- دیکشنری تلفظ ----------
def parse_replacements(text):
    pairs = []
    for ln in (text or '').splitlines():
        ln = ln.strip()
        if not ln or ln.startswith('#'):
            continue
        m = re.split(r'\s*(?:=>|->|=)\s*', ln, maxsplit=1)
        if len(m) == 2 and m[0]:
            pairs.append((m[0], m[1]))
    return pairs

def apply_replacements(text, pairs):
    for w, c in (pairs or []):
        text = text.replace(w, c)
    return text

# ---------- تقسیم به جمله ----------
def _split_long(s, max_chars):
    if len(s) <= max_chars:
        return [s]
    out, cur = [], ''
    for p in re.split(r'(?<=[،,])\s+', s):
        if len(p) > max_chars:
            for w in p.split(' '):
                if cur and len(cur) + 1 + len(w) > max_chars:
                    out.append(cur)
                    cur = w
                else:
                    cur = (cur + ' ' + w).strip()
            continue
        if cur and len(cur) + 1 + len(p) > max_chars:
            out.append(cur)
            cur = p
        else:
            cur = (cur + ' ' + p).strip()
    if cur:
        out.append(cur)
    return out

def split_paragraph(para, max_chars=200, min_chars=12):
    sents = []
    for s in re.split(r'(?<=[\.\!\?؟…؛])\s+', para):
        s = s.strip()
        if s:
            sents.extend(_split_long(s, max_chars))
    merged = []
    for s in sents:
        if merged and (len(merged[-1]) < min_chars or len(s) < min_chars) \
                and len(merged[-1]) + 1 + len(s) <= max_chars:
            merged[-1] = merged[-1] + ' ' + s
        else:
            merged.append(s)
    return merged

def build_units(paragraphs, replacements=None, max_chars=200):
    units = []
    for pi, para in enumerate(paragraphs):
        p = normalize_text(apply_replacements(para, replacements))
        sents = [s for s in split_paragraph(p, max_chars) if re.search(r'\w', s)]
        for si, s in enumerate(sents):
            units.append({'text': s, 'para': pi, 'last': si == len(sents) - 1})
    return units

# ---------- خواندن فایل ----------
def read_text_file(path):
    raw = open(path, 'rb').read()
    if raw.startswith((b'\xff\xfe', b'\xfe\xff')):
        return raw.decode('utf-16'), 'utf-16'
    if raw.startswith(b'\xef\xbb\xbf'):
        return raw.decode('utf-8-sig'), 'utf-8 (BOM)'
    try:
        return raw.decode('utf-8'), 'utf-8'
    except UnicodeDecodeError:
        pass
    if raw and raw.count(b'\x00') > len(raw) // 4:
        try:
            return raw.decode('utf-16-le'), 'utf-16'
        except UnicodeDecodeError:
            pass
    try:
        return raw.decode('cp1256'), 'windows-1256'
    except UnicodeDecodeError:
        return raw.decode('utf-8', errors='replace'), 'utf-8 (با جایگزینی کاراکترهای نامعتبر)'

def _clean_subtitle(text):
    lines = []
    for ln in text.splitlines():
        t = ln.strip()
        if not t or t.upper().startswith('WEBVTT') or '-->' in t or re.fullmatch(r'\d+', t):
            continue
        t = re.sub(r'<[^>]+>|\{[^}]+\}', '', t).strip()
        if t:
            lines.append(t)
    return [' '.join(lines)] if lines else []

def _clean_markdown(text):
    out = []
    for ln in text.splitlines():
        t = ln.strip()
        t = re.sub(r'^(#{1,6}|[-*+]|\d+\.)\s+', '', t)
        t = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', t)
        t = re.sub(r'[*_`~]', '', t).strip()
        if t:
            out.append(t)
    return out

SUPPORTED_EXTS = ('txt', 'md', 'docx', 'srt', 'vtt')

def load_paragraphs(path):
    ext = path.rsplit('.', 1)[-1].lower()
    if ext not in SUPPORTED_EXTS:
        raise ValueError(f'فرمت .{ext} پشتیبانی نمی‌شود. مجاز: {", ".join(SUPPORTED_EXTS)}')
    if ext == 'docx':
        import docx
        from docx.text.paragraph import Paragraph
        from docx.table import Table
        d = docx.Document(path)
        out = []
        for el in d.element.body.iterchildren():
            if el.tag.endswith('}p'):
                t = Paragraph(el, d).text.strip()
                if t:
                    out.append(t)
            elif el.tag.endswith('}tbl'):
                for row in Table(el, d).rows:
                    cells = []
                    for c in row.cells:
                        t = c.text.strip()
                        if t and t not in cells:
                            cells.append(t)
                    if cells:
                        out.append('، '.join(cells))
        return out, 'docx'
    text, enc = read_text_file(path)
    if ext in ('srt', 'vtt'):
        return _clean_subtitle(text), enc
    if ext == 'md':
        return _clean_markdown(text), enc
    return [ln.strip() for ln in text.splitlines() if ln.strip()], enc

print('✅ ابزارهای متن آماده شد.')

In [ ]:
#@title 📂 ۳) انتخاب فایل متنی
#@markdown فرمت‌های مجاز: txt, docx, md, srt, vtt
Use_Local_File = True #@param {type:"boolean"}
#@markdown اگر Use_Local_File غیرفعال است، مسیر فایل در Google Drive را اینجا بدهید:
Drive_File_Path = "/content/drive/MyDrive/text.txt" #@param {type:"string"}

SELECTED_DOC_PATH = None

if Use_Local_File:
    from google.colab import files
    print('📁 فایل متنی را انتخاب کنید...')
    up = files.upload()
    if not up:
        raise RuntimeError('❌ فایلی انتخاب نشد!')
    SELECTED_DOC_PATH = os.path.abspath(list(up.keys())[0])
else:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    SELECTED_DOC_PATH = Drive_File_Path.strip()
    if not os.path.exists(SELECTED_DOC_PATH):
        raise FileNotFoundError(f'❌ فایل پیدا نشد: {SELECTED_DOC_PATH}')

PARAGRAPHS, _enc = load_paragraphs(SELECTED_DOC_PATH)
if not PARAGRAPHS:
    raise RuntimeError('❌ متنی در فایل پیدا نشد!')

_chars = sum(len(p) for p in PARAGRAPHS)
print(f'✅ فایل: {os.path.basename(SELECTED_DOC_PATH)}  |  انکودینگ: {_enc}')
print(f'📄 {len(PARAGRAPHS)} پاراگراف، {_chars:,} کاراکتر')
print('— پیش‌نمایش (۳ پاراگراف اول) —')
for _p in PARAGRAPHS[:3]:
    print('  •', _p[:120] + ('…' if len(_p) > 120 else ''))
if '(' in _enc and 'جایگزینی' in _enc:
    print('⚠️ بعضی کاراکترهای فایل قابل خواندن نبود؛ فایل را با انکودینگ UTF-8 ذخیره کنید.')

In [ ]:
#@title 🎙️ ۴) آپلود نمونه صدا + متن نمونه
#@markdown نمونه‌ی صدا: فارسی، **۳ تا ۱۰ ثانیه**، بدون موسیقی و نویز.
#@markdown اگر فایل شما بلندتر است، با دو فیلد زیر یک بازه‌ی تمیز از آن انتخاب کنید (۰ یعنی «تا آخر فایل» / «از اول فایل»).
Ref_Start_Sec = 0 #@param {type:"number"}
Ref_End_Sec = 0 #@param {type:"number"}
#@markdown **متن دقیق همان بازه** را اینجا بنویسید (فقط چیزی که در بازه‌ی انتخابی گفته می‌شود، نه کل فایل).
#@markdown بدون متن، مدل با Whisper حدس می‌زند و برای فارسی ممکن است اشتباه کند.
Ref_Text = "" #@param {type:"string"}
#@markdown اگر فقط می‌خواهید بازه یا متن را عوض کنید، این را غیرفعال کنید تا فایل قبلی دوباره آپلود نشود:
Upload_New_File = True #@param {type:"boolean"}

import glob
from pydub import AudioSegment, effects
from pydub.silence import detect_leading_silence
from google.colab import files
from IPython.display import display, Audio

os.makedirs('/content/work', exist_ok=True)
_ORIG = '/content/work/ref_original'

if Upload_New_File or not glob.glob(_ORIG + '.*'):
    for _f in glob.glob('/content/work/ref_*'):
        os.remove(_f)
    print('🎙️ فایل صوتی نمونه را انتخاب کنید...')
    _up = files.upload()
    if not _up:
        raise RuntimeError('❌ فایلی آپلود نشد!')
    _fname = list(_up.keys())[0]
    _ext = _fname.rsplit('.', 1)[-1].lower() if '.' in _fname else 'wav'
    os.replace(_fname, f'{_ORIG}.{_ext}')

_orig_path = glob.glob(_ORIG + '.*')[0]
_full = AudioSegment.from_file(_orig_path).set_channels(1).set_frame_rate(24000)
print(f'📏 طول کل فایل آپلودشده: {len(_full)/1000:.1f} ثانیه')

# انتخاب بازه
_s_ms = int(max(0, Ref_Start_Sec) * 1000)
_e_ms = int(Ref_End_Sec * 1000) if Ref_End_Sec and Ref_End_Sec > 0 else len(_full)
if _e_ms <= _s_ms:
    raise ValueError('❌ Ref_End_Sec باید بزرگ‌تر از Ref_Start_Sec باشد.')
_seg = _full[_s_ms:_e_ms]

def _trim_edges(s, thr=-45, keep=80):
    a = detect_leading_silence(s, silence_threshold=thr)
    b = detect_leading_silence(s.reverse(), silence_threshold=thr)
    return s[max(0, a - keep): len(s) - max(0, b - keep)]

_seg = _trim_edges(_seg)
_dur = len(_seg) / 1000.0
REF_TEXT = normalize_text(Ref_Text) if Ref_Text.strip() else ''

if _dur > 20 and REF_TEXT:
    raise RuntimeError(
        f'❌ بازه‌ی انتخابی {_dur:.1f} ثانیه است (بیش از ۲۰ ثانیه) و کیفیت کلون خراب می‌شود.\n'
        '   یک بازه‌ی ۳ تا ۱۰ ثانیه‌ای با Ref_Start_Sec / Ref_End_Sec انتخاب کنید '
        'و متن دقیق همان بازه را در Ref_Text بنویسید (Upload_New_File را غیرفعال کنید).')
elif _dur > 10:
    if REF_TEXT:
        print(f'⚠️ بازه {_dur:.1f} ثانیه است؛ برای کیفیت و سرعت بهتر آن را به ۳ تا ۱۰ ثانیه کوتاه کنید '
              '(و متن را مطابق آن بنویسید).')
    else:
        _seg = _seg[:10000]
        _dur = 10.0
        print('✂️ چون متن نمونه خالی است، ۱۰ ثانیه‌ی اول بازه برداشته شد (Whisper همین بخش را می‌نویسد).')
elif _dur < 3:
    print(f'⚠️ نمونه فقط {_dur:.1f} ثانیه است؛ کلون ممکن است ضعیف باشد (۳ تا ۱۰ ثانیه ایده‌آل است).')

if _seg.dBFS < -35:
    print('⚠️ صدای نمونه خیلی آرام است؛ نرمال می‌شود، ولی بهتر است نمونه‌ی با کیفیت‌تری بدهید.')
_seg = effects.normalize(_seg)

REF_AUDIO_PATH = '/content/work/ref_voice.wav'
_seg.export(REF_AUDIO_PATH, format='wav')

print(f'✅ نمونه‌ی نهایی آماده شد ({len(_seg)/1000:.1f} ثانیه، مونو، ۲۴ کیلوهرتز).')
if REF_TEXT:
    print(f'📝 متن نمونه: {REF_TEXT}')
    print('   ← حتماً گوش بدهید و مطمئن شوید دقیقاً همین جمله در صدای زیر گفته می‌شود.')
else:
    print('📝 متن نمونه خالی است ← Whisper خودکار متن را تشخیص می‌دهد (دقت کمتر برای فارسی).')
display(Audio(REF_AUDIO_PATH))

In [ ]:
#@title ✏️ ۵) دیکشنری اصلاح تلفظ (اختیاری — فایل اصلی شما تغییر نمی‌کند)
# هر خط: «کلمه‌ی اشتباه => شکل درست». خط‌هایی که با # شروع شوند نادیده گرفته می‌شوند.
# بعد از ویرایش این سلول را اجرا کنید و سلول تولید را دوباره بزنید؛
# فقط جمله‌هایی که تغییر کرده‌اند دوباره ساخته می‌شوند.
REPLACEMENTS = """
# مثال:
# نکته => نُکته
# گوگل => گوگِل
"""
_n = len(parse_replacements(REPLACEMENTS))
print(f'✅ {_n} قانون تلفظ ثبت شد.')

In [ ]:
#@title 🧩 ۶) هسته‌ی تولید صدا (فقط یک‌بار اجرا شود)
import time, hashlib
import numpy as np
import soundfile as sf
import torch
from tqdm.auto import tqdm
from pydub import AudioSegment
from IPython.display import display, Audio
from google.colab import files

SR = 24000
MAX_RETRIES = 3

DEFAULT_SETTINGS = dict(
    steps=32, speed=1.0, guidance=0.0, seed=42, max_chars=200,
    pause_sentence=250, pause_paragraph=700,
    cache_on_drive=False, auto_download=True,
)

def _cache_dir(on_drive):
    if on_drive:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            drive.mount('/content/drive')
        d = '/content/drive/MyDrive/voice_clone_cache'
    else:
        d = '/content/vc_cache'
    os.makedirs(d, exist_ok=True)
    return d

def _synth_one(text, path, cfg, ref_text):
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            torch.manual_seed(int(cfg['seed']) + attempt * 101)
            kwargs = dict(text=text, ref_audio=REF_AUDIO_PATH,
                          num_step=int(cfg['steps']), speed=float(cfg['speed']))
            if ref_text:
                kwargs['ref_text'] = ref_text
            if cfg['guidance']:
                kwargs['guidance_scale'] = float(cfg['guidance'])
            with torch.inference_mode():
                audio = omni_model.generate(**kwargs)
            wav = audio[0]
            if not isinstance(wav, torch.Tensor):
                wav = torch.as_tensor(wav)
            wav = wav.detach().float().cpu()
            if wav.dim() > 1:
                wav = wav[0]
            arr = wav.numpy()
            dur = len(arr) / SR
            n = len(text)
            if dur < max(0.3, n / 40.0):
                raise ValueError(f'خروجی مشکوک کوتاه ({dur:.1f}s برای {n} کاراکتر)')
            if dur > n / 3.0 + 3.0:
                raise ValueError(f'خروجی مشکوک طولانی ({dur:.1f}s برای {n} کاراکتر)')
            sf.write(path, arr, SR)
            return True, None
        except Exception as e:
            last_err = f'{type(e).__name__}: {e}'
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return False, last_err

def run_pipeline(paragraphs, out_name, **kw):
    cfg = {**DEFAULT_SETTINGS, **kw}
    if 'omni_model' not in globals():
        raise RuntimeError('❌ ابتدا سلول ۱ (لود مدل) را اجرا کنید.')
    if 'REF_AUDIO_PATH' not in globals() or not os.path.exists(REF_AUDIO_PATH):
        raise RuntimeError('❌ ابتدا سلول ۴ (نمونه صدا) را اجرا کنید.')
    ref_text = globals().get('REF_TEXT', '')
    pairs = parse_replacements(globals().get('REPLACEMENTS', ''))

    units = build_units(paragraphs, pairs, int(cfg['max_chars']))
    if not units:
        raise RuntimeError('❌ بعد از پاک‌سازی متنی برای خواندن باقی نماند!')

    cache = _cache_dir(cfg['cache_on_drive'])
    ref_sig = hashlib.sha1(open(REF_AUDIO_PATH, 'rb').read()).hexdigest()[:10]
    base = '|'.join([ref_sig, ref_text, str(LOADED_MODEL), str(cfg['steps']),
                     str(cfg['speed']), str(cfg['guidance']), str(cfg['seed'])])

    total_chars = sum(len(u['text']) for u in units)
    print(f'🧾 {len(units)} جمله/قطعه، {total_chars:,} کاراکتر. مدل: {LOADED_MODEL}')

    done = cached = 0
    failed = []
    t0 = time.time()
    for idx, u in enumerate(tqdm(units, desc='تولید صدا')):
        key = hashlib.sha1((base + '|' + u['text']).encode('utf-8')).hexdigest()[:20]
        path = f'{cache}/{key}.wav'
        if os.path.exists(path) and os.path.getsize(path) > 1000:
            u['path'] = path
            cached += 1
            continue
        ok, err = _synth_one(u['text'], path, cfg, ref_text)
        if ok:
            u['path'] = path
            done += 1
        else:
            failed.append((idx + 1, u['text'], err))

    print(f'⏱️ {time.time()-t0:.0f} ثانیه | جدید: {done} | از کش: {cached} | ناموفق: {len(failed)}')
    if failed:
        print('⚠️ این قطعه‌ها ساخته نشدند و در خروجی نیستند:')
        with open('/content/failed_segments.txt', 'w', encoding='utf-8') as fh:
            for i, t, e in failed:
                print(f'   [{i}] {t[:60]}…  ← {e}')
                fh.write(f'[{i}] {t}\n    {e}\n')
        print('   (فهرست کامل: /content/failed_segments.txt) — سلول را دوباره اجرا کنید، فقط همین‌ها دوباره تلاش می‌شوند.')

    pieces = []
    for u in units:
        if not u.get('path'):
            continue
        a, _ = sf.read(u['path'], dtype='float32')
        if a.ndim > 1:
            a = a.mean(axis=1)
        pieces.append(a)
        gap = cfg['pause_paragraph'] if u['last'] else cfg['pause_sentence']
        pieces.append(np.zeros(int(SR * gap / 1000), dtype=np.float32))
    if not pieces:
        raise RuntimeError('❌ هیچ قطعه‌ای ساخته نشد؛ خطاها را در بالا ببینید.')

    full = np.clip(np.concatenate(pieces), -1.0, 1.0)
    wav_path = f'/content/{out_name}.wav'
    mp3_path = f'/content/{out_name}.mp3'
    sf.write(wav_path, full, SR)
    AudioSegment.from_wav(wav_path).export(mp3_path, format='mp3', bitrate='192k')
    os.remove(wav_path)

    mb = os.path.getsize(mp3_path) / 1e6
    print(f'🎉 آماده شد: {mp3_path}  ({len(full)/SR/60:.1f} دقیقه، {mb:.1f} MB)')
    if mb < 20:
        display(Audio(mp3_path))
    else:
        print('ℹ️ فایل بزرگ است؛ پخش درون‌نوت‌بوک غیرفعال شد، آن را دانلود کنید.')
    if cfg['auto_download']:
        files.download(mp3_path)
    return mp3_path

print('✅ هسته‌ی تولید آماده است.')

In [ ]:
#@title ▶️ ۷) شروع تولید: خواندن فایل با صدای کلون‌شده
Steps = 32 #@param {type:"slider", min:8, max:64, step:8}
Speed = 1.0 #@param {type:"slider", min:0.5, max:2.0, step:0.1}
#@markdown Guidance_Scale: مقدار ۰ یعنی پیش‌فرض مدل. (مدل فارسی در نمونه‌ی رسمی خودش مقدار ۳ را استفاده می‌کند.)
Guidance_Scale = 0 #@param {type:"number"}
#@markdown Seed ثابت = نتیجه‌ی تکرارپذیر و صدای یکنواخت‌تر بین جمله‌ها.
Seed = 42 #@param {type:"integer"}
Max_Chars_Per_Segment = 200 #@param {type:"slider", min:80, max:400, step:20}
Pause_Between_Sentences_ms = 250 #@param {type:"integer"}
Pause_Between_Paragraphs_ms = 700 #@param {type:"integer"}
#@markdown ذخیره‌ی کش روی Google Drive (برای متن‌های خیلی بلند، تا با قطع شدن Colab چیزی از دست نرود):
Cache_On_Drive = False #@param {type:"boolean"}
Output_Name = "cloned_voice" #@param {type:"string"}

if 'PARAGRAPHS' not in globals() or not PARAGRAPHS:
    raise RuntimeError('❌ ابتدا سلول ۳ (انتخاب فایل متنی) را اجرا کنید!')

SETTINGS = dict(
    steps=Steps, speed=Speed, guidance=Guidance_Scale, seed=Seed,
    max_chars=Max_Chars_Per_Segment,
    pause_sentence=Pause_Between_Sentences_ms,
    pause_paragraph=Pause_Between_Paragraphs_ms,
    cache_on_drive=Cache_On_Drive,
)
OUTPUT_FILE = run_pipeline(PARAGRAPHS, Output_Name.strip() or 'cloned_voice', **SETTINGS)

In [ ]:
#@title 🧪 (اختیاری) تست سریع با متن مستقیم
#@markdown چند جمله بنویسید تا قبل از پردازش فایل اصلی، کیفیت صدا و تلفظ را بسنجید.
Manual_Text = "سلام، این یک آزمایش کوتاه است. امروز ۲۰ آذر ۱۴۰۵ و دمای هوا ۱۲٫۵ درجه است." #@param {type:"string"}

_paras = [ln.strip() for ln in Manual_Text.splitlines() if ln.strip()]
if not _paras:
    raise RuntimeError('❌ متنی وارد نشده است!')
_s = dict(globals().get('SETTINGS', {}))
_s['auto_download'] = False
run_pipeline(_paras, 'cloned_voice_test', **_s)

In [ ]:
#@title 🧹 پاکسازی کش و فایل‌های موقت (خروجی‌های mp3 و فایل‌های شما حذف نمی‌شوند)
Confirm_Delete = False #@param {type:"boolean"}
import shutil
if not Confirm_Delete:
    print('ℹ️ برای پاک کردن، تیک Confirm_Delete را بزنید.')
else:
    for d in ['/content/vc_cache', '/content/work']:
        shutil.rmtree(d, ignore_errors=True)
    print('✅ کش محلی و نمونه‌ی صدای آماده‌شده پاک شد. (کش Google Drive در صورت استفاده دست‌نخورده است.)')

## 🏷️ تگ‌های حالت صوتی (برخی ضعیف عمل می‌کنند)

تگ را مستقیم داخل متن بنویسید، مثلاً: `این خیلی جالب بود [laughter] واقعاً!`

| تگ | واکنش صوتی |
|---|---|
| `[laughter]` | خنده |
| `[sigh]` | آه کشیدن |
| `[confirmation-en]` | صدای تأییدی (هوم) |
| `[question-en]` `[question-ah]` `[question-oh]` `[question-ei]` `[question-yi]` | صدای سؤالی |
| `[surprise-ah]` `[surprise-oh]` `[surprise-wa]` `[surprise-yo]` | صدای تعجب |
| `[dissatisfaction-hnn]` | صدای نارضایتی |

## 💡 نکات کیفیت
- اگر کلمه‌ای غلط خوانده شد، در سلول ۵ شکل درست (با اعراب) را بنویسید؛ لازم نیست فایل متن را دستکاری کنید.
- اعداد، درصد و اعشار خودکار به حروف فارسی تبدیل می‌شوند. تاریخ‌های مثل `1405/09/20` را بهتر است خودتان به صورت حرفی بنویسید.
- اگر صدا بین جمله‌ها کمی تغییر کرد، `Seed` را ثابت نگه دارید و نمونه‌ی صدای تمیزتری بدهید.